# Tenrec QB-video：多路召回与真实负样本

QB-video 是曝光级日志：每行代表一次用户–视频曝光，`click=1` 是正样本，`click=0` 是真实曝光未点击负样本。ItemCF、UserCF、类目召回只使用点击正反馈；双塔 BPR 优先使用同一用户真实曝光未点击的视频作为负样本。

In [ ]:
# %pip install -q pandas pyarrow scipy implicit tqdm torch
from __future__ import annotations
import os, random
from collections import defaultdict
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import sparse
from tqdm.auto import tqdm
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from implicit.nearest_neighbours import BM25Recommender, CosineRecommender

SEED = 20260715
MIN_POSITIVE_CLICKS = 5
RECALL_K = 100
EVAL_USERS = 2_000
SIMILAR_K = 100
LATENT_DIM, BATCH_SIZE, EPOCHS, LR = 64, 4096, 5, 2e-3
RAW_DATA_PATH = Path(os.environ.get('TENREC_QB_VIDEO_PATH', '/root/shared-storage/Tenrec_QB_video/Tenrec/QB-video.csv'))
PROJECT_ROOT = Path(os.environ.get('TENREC_QB_PROJECT_ROOT', '/root/workspace/tenrec_qb_video_multirecall'))
for p in (PROJECT_ROOT / 'data', PROJECT_ROOT / 'models', PROJECT_ROOT / 'outputs'):
    p.mkdir(parents=True, exist_ok=True)
DATA_DIR, MODEL_DIR, OUTPUT_DIR = PROJECT_ROOT / 'data', PROJECT_ROOT / 'models', PROJECT_ROOT / 'outputs'
assert RAW_DATA_PATH.exists(), f'请修改 RAW_DATA_PATH：{RAW_DATA_PATH}'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('data:', RAW_DATA_PATH, '| device:', DEVICE)

## 1. 读取曝光日志并检查真实负样本

QB-video 仅约 244 万行，可在单机内存中读取。`event_order` 保留原始行顺序，作为无时间戳条件下的近似时间顺序。

In [ ]:
raw = pd.read_csv(RAW_DATA_PATH, low_memory=False)
raw['click'] = pd.to_numeric(raw['click'], errors='coerce').fillna(0).astype('int8')
raw['event_order'] = np.arange(len(raw), dtype=np.int64)
raw['user_id'] = raw['user_id'].astype('int64'); raw['item_id'] = raw['item_id'].astype('int64')
display(pd.DataFrame([{
    'exposures': len(raw), 'click_positives': int((raw.click == 1).sum()),
    'true_negatives_click_0': int((raw.click == 0).sum()),
    'users': raw.user_id.nunique(), 'items': raw.item_id.nunique(),
    'observed_ctr': float(raw.click.mean()),
}]))
display(raw.head())

## 2. 时间顺序切分与训练期真实负样本

每位用户最后一次点击留作测试、倒数第二次留作验证，之前点击用于训练。每位用户的真实负样本仅保留在其训练截止点之前发生的 `click=0` 曝光，避免看见未来曝光。

In [ ]:
positives = raw[raw.click == 1].sort_values(['user_id', 'event_order'], kind='stable').copy()
n_clicks = positives.groupby('user_id')['item_id'].transform('size')
eligible = positives[n_clicks >= MIN_POSITIVE_CLICKS].copy()
eligible['rank_from_end'] = eligible.groupby('user_id').cumcount(ascending=False)
test_df = eligible[eligible.rank_from_end == 0][['user_id', 'item_id']].copy()
val_df = eligible[eligible.rank_from_end == 1][['user_id', 'item_id']].copy()
train_raw = eligible[eligible.rank_from_end >= 2].copy()

# 行为越深，正反馈权重越高；QB-video 没有文章阅读时长/质量分。
for c in ['follow', 'like', 'share']:
    train_raw[c] = pd.to_numeric(train_raw[c], errors='coerce').fillna(0)
train_raw['weight'] = 1.0 + 3.0 * train_raw['like'] + 4.0 * train_raw['share'] + 4.0 * train_raw['follow']
train_df = (train_raw.sort_values('event_order').groupby(['user_id', 'item_id'], as_index=False)
            .agg(weight=('weight', 'max'), video_category=('video_category', 'last'),
                 gender=('gender', 'last'), age=('age', 'last'), event_order=('event_order', 'max')))

cutoff = train_raw.groupby('user_id', as_index=False)['event_order'].max().rename(columns={'event_order': 'train_cutoff'})
neg_exposures = raw[raw.click == 0].merge(cutoff, on='user_id', how='inner')
neg_exposures = neg_exposures[neg_exposures.event_order <= neg_exposures.train_cutoff]
seen_positive = {u: set(f.item_id) for u, f in train_df.groupby('user_id')}
negative_by_user = {u: set(f.item_id) - seen_positive.get(u, set()) for u, f in neg_exposures.groupby('user_id')}
print({'train_positive_pairs': len(train_df), 'validation': len(val_df), 'test': len(test_df),
       'users_with_real_negatives': sum(bool(x) for x in negative_by_user.values())})

In [ ]:
user_ids, item_ids = np.sort(train_df.user_id.unique()), np.sort(train_df.item_id.unique())
u2i = {u: i for i, u in enumerate(user_ids)}; it2i = {it: i for i, it in enumerate(item_ids)}
rows, cols = train_df.user_id.map(u2i).to_numpy(), train_df.item_id.map(it2i).to_numpy()
user_item = sparse.csr_matrix((train_df.weight.astype('float32'), (rows, cols)), shape=(len(user_ids), len(item_ids)))
truth = dict(zip(test_df.user_id, test_df.item_id))
eval_users = np.random.default_rng(SEED).choice(np.array(sorted(truth)), min(EVAL_USERS, len(truth)), replace=False)

def candidate_frame(rows, source):
    return pd.DataFrame(rows, columns=['user_id', 'item_id', 'score']).assign(source=source)
def recall_at_k(df, k=RECALL_K):
    top = df.sort_values(['user_id', 'score'], ascending=[True, False]).groupby('user_id').head(k)
    hit_users = set(top.loc[top.apply(lambda r: truth.get(int(r.user_id)) == int(r.item_id), axis=1), 'user_id'])
    return len(hit_users) / len(eval_users)


## 3. ItemCF、UserCF、视频类目召回

这三路只把点击视为正反馈，不把 `click=0` 当作相似度图中的正边。

In [ ]:
itemcf = BM25Recommender(K=SIMILAR_K); itemcf.fit(user_item.T.tocsr(), show_progress=True)
usercf = CosineRecommender(K=SIMILAR_K); usercf.fit(user_item.tocsr(), show_progress=True)

item_meta = train_df.groupby('item_id', as_index=False).agg(video_category=('video_category', 'last'))
tag_pop = (train_df.groupby(['video_category', 'item_id'], as_index=False).weight.sum()
           .sort_values(['video_category', 'weight'], ascending=[True, False]).groupby('video_category').head(500))
items_by_tag = {t: list(zip(f.item_id, f.weight)) for t, f in tag_pop.groupby('video_category')}
user_tag = (train_df.groupby(['user_id', 'video_category'], as_index=False).weight.sum()
            .sort_values(['user_id', 'weight'], ascending=[True, False]).groupby('user_id').head(5))

def itemcf_recall(u, n=RECALL_K):
    ui = u2i[u]; ids, scores = itemcf.recommend(ui, user_item, N=n, filter_already_liked_items=True)
    return [(u, int(item_ids[i]), float(s)) for i, s in zip(ids, scores)]
def usercf_recall(u, n=RECALL_K):
    ui, scores = u2i[u], defaultdict(float)
    ns, ss = usercf.similar_items(ui, N=SIMILAR_K + 1)
    for ni, sim in zip(ns, ss):
        if int(ni) == ui: continue
        start, end = user_item.indptr[int(ni)], user_item.indptr[int(ni)+1]
        for ii, w in zip(user_item.indices[start:end], user_item.data[start:end]):
            item = int(item_ids[ii])
            if item not in seen_positive[u]: scores[item] += float(sim) * float(w)
    return [(u, item, score) for item, score in sorted(scores.items(), key=lambda x: x[1], reverse=True)[:n]]
def tag_recall(u, n=RECALL_K):
    scores = defaultdict(float); seen = seen_positive[u]
    for _, r in user_tag[user_tag.user_id == u].iterrows():
        for item, pop in items_by_tag.get(r.video_category, []):
            if item not in seen: scores[int(item)] += float(r.weight) * float(pop)
    return [(u, item, score) for item, score in sorted(scores.items(), key=lambda x: x[1], reverse=True)[:n]]

itemcf_rows=[]; usercf_rows=[]; tag_rows=[]
for u in tqdm(eval_users, desc='生成 CF/类目候选'):
    u=int(u); itemcf_rows += itemcf_recall(u); usercf_rows += usercf_recall(u); tag_rows += tag_recall(u)
itemcf_result=candidate_frame(itemcf_rows,'itemcf'); usercf_result=candidate_frame(usercf_rows,'usercf'); tag_result=candidate_frame(tag_rows,'video_category')
for result in [itemcf_result, usercf_result, tag_result]: print(result.source.iloc[0], 'Recall@%d' % RECALL_K, recall_at_k(result))

## 4. 双塔：优先使用真实曝光负样本的 BPR 训练

若某用户训练期没有 `click=0` 曝光，则退回到未点击正样本集合中的均匀负采样。

In [ ]:
def code_map(s): return {v:i+1 for i,v in enumerate(sorted(s.dropna().astype(str).unique()))}
gender_map, age_map, category_map = code_map(train_df.gender), code_map(train_df.age), code_map(item_meta.video_category)
uf = pd.DataFrame({'user_id':user_ids}).merge(train_df.groupby('user_id',as_index=False).agg(gender=('gender','last'),age=('age','last')),on='user_id',how='left')
itf = pd.DataFrame({'item_id':item_ids}).merge(item_meta,on='item_id',how='left')
ug=torch.tensor(uf.gender.astype(str).map(gender_map).fillna(0).astype('int64').to_numpy(),device=DEVICE)
ua=torch.tensor(uf.age.astype(str).map(age_map).fillna(0).astype('int64').to_numpy(),device=DEVICE)
ic=torch.tensor(itf.video_category.astype(str).map(category_map).fillna(0).astype('int64').to_numpy(),device=DEVICE)

class TwoTower(nn.Module):
    def __init__(self):
        super().__init__(); self.ue=nn.Embedding(len(user_ids),LATENT_DIM); self.ge=nn.Embedding(len(gender_map)+1,8); self.ae=nn.Embedding(len(age_map)+1,8); self.ie=nn.Embedding(len(item_ids),LATENT_DIM); self.ce=nn.Embedding(len(category_map)+1,16)
        self.um=nn.Sequential(nn.Linear(LATENT_DIM+16,128),nn.ReLU(),nn.Linear(128,LATENT_DIM)); self.im=nn.Sequential(nn.Linear(LATENT_DIM+16,128),nn.ReLU(),nn.Linear(128,LATENT_DIM))
    def user(self,u): return nn.functional.normalize(self.um(torch.cat([self.ue(u),self.ge(ug[u]),self.ae(ua[u])],-1)),dim=-1)
    def item(self,i): return nn.functional.normalize(self.im(torch.cat([self.ie(i),self.ce(ic[i])],-1)),dim=-1)

neg_pool_idx={u:np.array([it2i[x] for x in xs if x in it2i],dtype=np.int64) for u,xs in negative_by_user.items()}
positive_idx_by_user={u:set(g.item_id.map(it2i).to_numpy(np.int64)) for u,g in train_df.groupby('user_id')}
train_u=train_df.user_id.map(u2i).to_numpy(np.int64); train_i=train_df.item_id.map(it2i).to_numpy(np.int64); train_w=np.log1p(train_df.weight.to_numpy(np.float32))
loader=DataLoader(TensorDataset(torch.from_numpy(train_u),torch.from_numpy(train_i),torch.from_numpy(train_w)),batch_size=BATCH_SIZE,shuffle=True,num_workers=2,pin_memory=torch.cuda.is_available())
model=TwoTower().to(DEVICE); opt=torch.optim.AdamW(model.parameters(),lr=LR); rng=np.random.default_rng(SEED)
for epoch in range(EPOCHS):
    total=0.0
    for u,pos,w in tqdm(loader,desc=f'epoch {epoch+1}',leave=False):
        u_np=u.numpy(); neg_np=np.empty(len(u_np),dtype=np.int64)
        for j,uidx in enumerate(u_np):
            raw_uid=int(user_ids[uidx]); pool=neg_pool_idx.get(raw_uid)
            if pool is not None and len(pool): neg_np[j]=rng.choice(pool)
            else:
                seen=positive_idx_by_user.get(raw_uid,set()); candidate=rng.integers(len(item_ids))
                for _ in range(20):
                    if candidate not in seen: break
                    candidate=rng.integers(len(item_ids))
                neg_np[j]=candidate
        u,pos,w=u.to(DEVICE),pos.to(DEVICE),w.to(DEVICE); neg=torch.tensor(neg_np,device=DEVICE)
        loss=(-nn.functional.logsigmoid((model.user(u)*model.item(pos)).sum(-1)-(model.user(u)*model.item(neg)).sum(-1))*w).mean()
        opt.zero_grad(); loss.backward(); opt.step(); total += float(loss)*len(u)
    print('epoch',epoch+1,'loss',total/len(train_df))
torch.save(model.state_dict(),MODEL_DIR/'two_tower_qb_video.pt')

In [ ]:
@torch.no_grad()
def tower_recall(u,n=RECALL_K):
    model.eval(); all_i=torch.arange(len(item_ids),device=DEVICE); score=(model.user(torch.tensor([u2i[u]],device=DEVICE))@model.item(all_i).T).squeeze()
    seen=[it2i[x] for x in seen_positive[u] if x in it2i]; score[torch.tensor(seen,device=DEVICE)]=-torch.inf
    s,idx=torch.topk(score,n); return [(u,int(item_ids[i]),float(v)) for i,v in zip(idx.cpu().numpy(),s.cpu().numpy())]
tower_rows=[]
for u in tqdm(eval_users,desc='生成双塔候选'): tower_rows += tower_recall(int(u))
tower_result=candidate_frame(tower_rows,'two_tower_true_negative'); print('two_tower Recall@%d' % RECALL_K,recall_at_k(tower_result))

all_candidates=pd.concat([itemcf_result,usercf_result,tag_result,tower_result],ignore_index=True)
all_candidates['rank_in_source']=all_candidates.groupby(['source','user_id']).score.rank(method='first',ascending=False)
all_candidates.to_parquet(OUTPUT_DIR/'candidates_all_sources.parquet',index=False)
display(all_candidates.groupby('source').size().rename('candidate_rows').to_frame())